# Approval Bypass Detection Notebook

This notebook identifies purchase orders approved above the approver's authorization limit, recorded without an approver, assigned to an inactive approver, or assigned to an approver not found in the employee master. The logic mirrors SAP MM release strategy governance and workflow agent controls.

## 1. Load PO and employee data

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", palette="Set2")
pd.options.display.float_format = "{:,.0f}".format

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "03_Data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "03_Data"
OUTPUT_DIR = PROJECT_ROOT / "05_Python_Analysis" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def yen(value):
    return f"¥{value:,.0f}"

purchase_orders = pd.read_csv(DATA_DIR / "purchase_orders.csv", parse_dates=["po_date", "approval_date"])
employees = pd.read_csv(DATA_DIR / "employees.csv")
vendors = pd.read_csv(DATA_DIR / "vendors.csv")

po_data = purchase_orders.merge(vendors[["vendor_id", "vendor_name", "vendor_category"]], on="vendor_id", how="left")
po_data.head()

## 2. Map approval limits to employees

In [ ]:
employee_limits = employees.rename(columns={
    "employee_id": "approved_by",
    "employee_name": "approver_name",
    "department": "approver_department",
    "approval_limit": "employee_approval_limit",
    "approval_level": "employee_approval_level",
    "is_active": "approver_is_active",
})

approved_po_data = po_data.merge(employee_limits, on="approved_by", how="left")
approved_po_data["employee_approval_limit"] = approved_po_data["employee_approval_limit"].fillna(0)
approved_po_data[["po_number", "total_amount", "approved_by", "approver_name", "employee_approval_limit"]].head()

## 3. Detect approval control exceptions

In [ ]:
# SAP business logic: a valid release strategy should route each PO to an active approver with sufficient authority.
approved_po_data["missing_approval"] = approved_po_data["approved_by"].isna() | approved_po_data["approved_by"].astype(str).str.strip().eq("")
approved_po_data["approver_not_found"] = (~approved_po_data["missing_approval"]) & approved_po_data["approver_name"].isna()
approved_po_data["employee_approval_limit"] = approved_po_data["employee_approval_limit"].fillna(0)
approved_po_data["inactive_approver"] = (
    ~approved_po_data["missing_approval"]
    & ~approved_po_data["approver_not_found"]
    & ~approved_po_data["approver_is_active"].astype(str).eq("True")
)
approved_po_data["limit_violation"] = (
    ~approved_po_data["missing_approval"]
    & ~approved_po_data["approver_not_found"]
    & (approved_po_data["total_amount"] > approved_po_data["employee_approval_limit"])
)
approved_po_data["approval_bypass_detected"] = (
    approved_po_data["missing_approval"]
    | approved_po_data["approver_not_found"]
    | approved_po_data["inactive_approver"]
    | approved_po_data["limit_violation"]
)
approved_po_data["excess_amount_jpy"] = 0
full_value_exception = approved_po_data["missing_approval"] | approved_po_data["approver_not_found"] | approved_po_data["inactive_approver"]
approved_po_data.loc[full_value_exception, "excess_amount_jpy"] = approved_po_data.loc[full_value_exception, "total_amount"]
approved_po_data.loc[approved_po_data["limit_violation"], "excess_amount_jpy"] = (
    approved_po_data.loc[approved_po_data["limit_violation"], "total_amount"]
    - approved_po_data.loc[approved_po_data["limit_violation"], "employee_approval_limit"]
)
approved_po_data["bypass_value_jpy"] = approved_po_data["excess_amount_jpy"]
approved_po_data["violation_reason"] = approved_po_data.apply(
    lambda row: "Missing approval" if row["missing_approval"]
    else "Approver not found" if row["approver_not_found"]
    else "Inactive approver" if row["inactive_approver"]
    else "Amount exceeds approver limit" if row["limit_violation"]
    else "No violation",
    axis=1,
)

approval_violations = approved_po_data[approved_po_data["approval_bypass_detected"]].copy()
approval_violations = approval_violations.sort_values(["excess_amount_jpy", "total_amount", "po_date"], ascending=[False, False, False])
approval_violations.to_csv(OUTPUT_DIR / "approval_bypass_violations.csv", index=False)
approval_violations.head(25)


## 4. Analyze exception exposure by department

In [ ]:
department_risk = (
    approval_violations.assign(approver_department=approval_violations["approver_department"].fillna("Unassigned / Missing"))
    .groupby("approver_department", as_index=False)
    .agg(
        bypass_count=("po_number", "count"),
        total_bypassed_value_jpy=("total_amount", "sum"),
        total_excess_amount_jpy=("excess_amount_jpy", "sum"),
        average_bypass_value_jpy=("total_amount", "mean"),
        max_bypass_value_jpy=("total_amount", "max"),
    )
    .sort_values("total_excess_amount_jpy", ascending=False)
)
department_risk.to_csv(OUTPUT_DIR / "approval_bypass_department_risk.csv", index=False)
department_risk


In [ ]:
plt.figure(figsize=(11, 6))
sns.barplot(data=department_risk, y="approver_department", x="total_excess_amount_jpy", color="crimson")
plt.title("Departments with Highest Approval Bypass Excess Exposure")
plt.xlabel("Excess approval value (JPY)")
plt.ylabel("Department")
plt.tight_layout()
plt.show()


## 5. Identify employees with most bypasses

In [ ]:
employee_bypass_ranking = (
    approval_violations.assign(approver_name=approval_violations["approver_name"].fillna("Missing approver"))
    .groupby(["approved_by", "approver_name", "approver_department"], dropna=False, as_index=False)
    .agg(
        bypass_count=("po_number", "count"),
        total_bypassed_value_jpy=("total_amount", "sum"),
        total_excess_amount_jpy=("excess_amount_jpy", "sum"),
        average_bypass_value_jpy=("total_amount", "mean"),
        highest_bypass_value_jpy=("total_amount", "max"),
    )
    .sort_values(["bypass_count", "total_excess_amount_jpy"], ascending=False)
)
employee_bypass_ranking.to_csv(OUTPUT_DIR / "approval_bypass_employee_ranking.csv", index=False)
employee_bypass_ranking.head(20)


In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=employee_bypass_ranking.head(10), y="approver_name", x="bypass_count", color="purple")
plt.title("Employees with Most Approval Bypasses")
plt.xlabel("Bypass count")
plt.ylabel("Approver")
plt.tight_layout()
plt.show()

## 6. Timeline of bypass activity

In [ ]:
bypass_timeline = (
    approval_violations.assign(month=approval_violations["po_date"].dt.to_period("M").dt.to_timestamp())
    .groupby("month", as_index=False)
    .agg(bypass_count=("po_number", "count"), bypass_value_jpy=("total_amount", "sum"), excess_amount_jpy=("excess_amount_jpy", "sum"))
)

fig, ax1 = plt.subplots(figsize=(12, 5))
sns.lineplot(data=bypass_timeline, x="month", y="bypass_count", marker="o", ax=ax1, label="Bypass count")
ax1.set_ylabel("Bypass count")
ax2 = ax1.twinx()
sns.lineplot(data=bypass_timeline, x="month", y="excess_amount_jpy", marker="s", color="crimson", ax=ax2, label="Excess value")
ax2.set_ylabel("Excess value (JPY)")
ax1.set_title("Approval Bypass Activity Over Time")
fig.tight_layout()
plt.show()


In [ ]:
approval_summary = pd.DataFrame({
    "metric": ["approval_violations", "total_value_of_bypassed_approvals_jpy", "total_excess_approval_value_jpy", "departments_impacted", "employees_impacted"],
    "value": [
        len(approval_violations),
        approval_violations["total_amount"].sum(),
        approval_violations["excess_amount_jpy"].sum(),
        approval_violations["approver_department"].nunique(dropna=True),
        approval_violations["approved_by"].nunique(dropna=True),
    ],
})
approval_summary
